In [1]:
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126
!pip install torch_geometric pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.8/821.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 5.3 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.5.0
    Uninstalling triton-3.5.0:
      Successfully uninstalled triton-3.5.0
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.27.5
    Uninstalling nvidia-nccl-cu12-2.27.5:
      Successfully uninstalled nvidia-nccl-cu12-2.27.5
  Attempting uninstall: torch
    Found existing installation: torch 2.9.0+cu126
    Uninstalling torch-2.9.0+cu126:
      Successfully uninstalled torch-2.9.0+cu126
  Attempting uninstall: torchvision

In [2]:
import torch
import optuna
import torch.nn as nn
import torch.nn.functional as F
import json

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from sklearn import metrics

In [3]:
class GIN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout):
        super().__init__()

        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()

        for _ in range(num_layers):
            self.convs.append(
                GINConv(nn.Sequential(
                    nn.Linear(input_dim, 2 * hidden_dim),
                    nn.BatchNorm1d(2 * hidden_dim),
                    nn.ReLU(),
                    nn.Linear(2 * hidden_dim, hidden_dim),
                ))
            )
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
            input_dim = hidden_dim

        self.lin1 = nn.Linear(hidden_dim, hidden_dim)
        self.batch_norm1 = nn.BatchNorm1d(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, batch_norm in zip(self.convs, self.batch_norms):
            x = F.relu(batch_norm(conv(x, edge_index)))
            x = F.dropout(x, self.dropout, training=self.training)
        x = global_add_pool(x, batch)
        x = F.relu(self.batch_norm1(self.lin1(x)))
        return self.classifier(x).view(-1)

In [ ]:
train_dataset = torch.load("/kaggle/input/datasets/samuelvarchol/graphs-with-autgrpg-1/train_dataset_baseline.pt",weights_only=False)
val_dataset = torch.load("/kaggle/input/datasets/samuelvarchol/graphs-with-autgrpg-1/val_dataset_baseline.pt",weights_only=False)

number_of_features = train_dataset[0].num_node_features

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
def objective(trial: optuna.Trial):
    global model, optimizer, criterion, scheduler, train_loader, val_loader

    hidden_dim = trial.suggest_categorical("hidden_dim", [64, 128, 256, 512])
    num_layers = trial.suggest_int("num_layers", 2, 6)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    model = GIN(number_of_features, hidden_dim=hidden_dim, num_layers=num_layers, dropout=dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
    )

    n_epochs = 50
    best_val_acc = 0.0
    patience = 10
    patience_counter = 0

    trial_history = []
    
    for epoch in range(1, n_epochs + 1):
        train_loss = train()
        train_acc, train_f1 = test(train_loader)
        val_acc, val_f1 = test(val_loader)
        
        scheduler.step(val_acc)

        trial_history.append({
            "trial": trial.number,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "lr": optimizer.param_groups[0]["lr"]
        })
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1
        
        trial.report(val_acc, epoch)
        
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        if patience_counter >= patience:
            print(f"Trial {trial.number}: Early stopping at epoch {epoch}")
            break
        
        if epoch % 10 == 0:
            print(f"Trial {trial.number} | Epoch {epoch:02d} | "
                  f"Train Loss: {train_loss:.4f} | "
                  f"Val Acc: {val_acc:.4f}")
    
    return best_val_acc

In [6]:
def train():
    model.train()
    total_loss = 0

    for data in train_loader:
        data = data.to(device)
        out = model(data)
        loss = criterion(out, data.y.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.num_graphs
        
    return total_loss / len(train_loader.dataset)


@torch.no_grad()
def test(loader):
    model.eval()
    predictions = []
    labels = []

    for data in loader:
        data = data.to(device)
        out = model(data)
        pred = (out > 0).float()
        predictions.append(pred.cpu())
        labels.append(data.y.cpu())

    accuracy = metrics.accuracy_score(torch.cat(labels), torch.cat(predictions))
    f1 = metrics.f1_score(torch.cat(labels), torch.cat(predictions))

    return accuracy, f1

In [7]:
study = optuna.create_study(
    direction="maximize", 
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
    study_name="GIN for partial automorphism extension problem with baseline dataset")

study.optimize(objective, n_trials=100)

trials_df = study.trials_dataframe()
trials_df.to_csv("/kaggle/working/optuna_trials_summary.csv", index=False)

trial = study.best_trial
print(f"  Value (Val Acc): {trial.value:.4f}")
print("\n  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

best_params = study.best_params
config_path = "/kaggle/working/best_config.json"
with open(config_path, "w") as f:
    json.dump(best_params, f, indent=4)

[I 2026-03-03 17:35:05,673] A new study created in memory with name: GIN for partial automorphism extension problem with baseline dataset


Trial 0 | Epoch 10 | Train Loss: 0.4472 | Val Acc: 0.7715
Trial 0 | Epoch 20 | Train Loss: 0.3690 | Val Acc: 0.7880
Trial 0 | Epoch 30 | Train Loss: 0.3072 | Val Acc: 0.7994
Trial 0 | Epoch 40 | Train Loss: 0.2682 | Val Acc: 0.8033


[I 2026-03-03 17:41:24,618] Trial 0 finished with value: 0.8084630601186893 and parameters: {'hidden_dim': 512, 'num_layers': 3, 'dropout': 0.02589413555344877, 'lr': 0.00011789914511057849, 'weight_decay': 0.00013176095108210839, 'batch_size': 256}. Best is trial 0 with value: 0.8084630601186893.


Trial 0 | Epoch 50 | Train Loss: 0.2419 | Val Acc: 0.8039
Trial 1 | Epoch 10 | Train Loss: 0.5306 | Val Acc: 0.7386
Trial 1 | Epoch 20 | Train Loss: 0.4993 | Val Acc: 0.7518
Trial 1 | Epoch 30 | Train Loss: 0.4636 | Val Acc: 0.7709
Trial 1 | Epoch 40 | Train Loss: 0.4329 | Val Acc: 0.7845


[I 2026-03-03 17:53:50,407] Trial 1 finished with value: 0.7909176915799432 and parameters: {'hidden_dim': 512, 'num_layers': 5, 'dropout': 0.11100591416240918, 'lr': 0.007248699883248075, 'weight_decay': 0.0005098291941068777, 'batch_size': 64}. Best is trial 0 with value: 0.8084630601186893.


...

Trial 96 | Epoch 10 | Train Loss: 0.4665 | Val Acc: 0.7715
Trial 96 | Epoch 20 | Train Loss: 0.4092 | Val Acc: 0.7905


[I 2026-03-04 00:12:13,118] Trial 96 pruned. 
[I 2026-03-04 00:13:55,716] Trial 97 pruned. 


Trial 98 | Epoch 10 | Train Loss: 0.4712 | Val Acc: 0.7726
Trial 98 | Epoch 20 | Train Loss: 0.4296 | Val Acc: 0.7851


[I 2026-03-04 00:18:48,376] Trial 98 pruned. 
[I 2026-03-04 00:20:47,631] Trial 99 pruned. 


  Value (Val Acc): 0.8163

  Params: 
    hidden_dim: 512
    num_layers: 3
    dropout: 0.0541996739358364
    lr: 0.00011899765906637908
    weight_decay: 6.700674460021161e-05
    batch_size: 64
